# Gundam Tracking Project

In [1]:
import requests
import pandas as pd
from datetime import datetime

gundam_category = '86'

r = requests.get(f"https://tcgcsv.com/tcgplayer/{gundam_category}/groups")
all_groups = r.json()['results']
product_df = pd.DataFrame()
product_rows = []
price_rows = []

for group in all_groups:
    #Group ID = TCGPlayer's Page (ie. GD01, GD02, ST01)
    group_id = group['groupId']
    r = requests.get(f"https://tcgcsv.com/tcgplayer/{gundam_category}/{group_id}/products")
    products = r.json()['results']

    for product in products:
        # Process product information
        product_row_data = {
            'productId': product['productId'],
            'name': product['name'],
            'cleanName': product['cleanName'],
            'imageUrl': product['imageUrl'],
            'categoryId':product['categoryId'],
            'groupId':product['groupId'],
            'url':product['url'],
            'modifiedOn': product['modifiedOn'],
            'imageCount': product['imageCount'],
            'isPresale':product['presaleInfo'].get('isPresale'),
            'releasedOn':product['presaleInfo'].get('releasedOn'),
        }
        
        #Break down the nested dictionary within the data
        for field in product.get('extendedData',[]): #in another list
            col_name = field['name']
            product_row_data[col_name] = field['value']
        
        product_rows.append(product_row_data)
        #print(f"{product['productId']} - {product['name']}")
        

    r = requests.get(f"https://tcgcsv.com/tcgplayer/{gundam_category}/{group_id}/prices")
    prices = r.json()['results']

    for price in prices:
        # Process prices
        price_row_data = {
            'productId':price['productId'],
            'lowPrice':price['lowPrice'],
            'midPrice':price['midPrice'],
            'highPrice':price['highPrice'],
            'marketPrice':price['marketPrice'],
            'directLowPrice':price['directLowPrice'],
            'subTypeName':price['subTypeName']
        }
        
        price_rows.append(price_row_data)
        #print(f"{price['productId']} - {price['subTypeName']} - {price['midPrice']}")

    #break  (Only process the first group and break for testing)
    

In [2]:
#Check when was the data was last updated
date_request = requests.get('https://tcgcsv.com/last-updated.txt').text
format_string = '%Y-%m-%dT%H:%M:%S%z'
date = datetime.strptime(date_request, format_string).replace(tzinfo=None).date()
str(date)

'2026-01-29'

In [3]:
#Import scraped data into dataframes for cleaning/exploration
product_df = pd.DataFrame(product_rows)
price_df = pd.DataFrame(price_rows)

In [4]:
#Add the date into both dataframes for tracking
product_df['Date'] = date
price_df['Date'] = date

In [5]:
product_df.head()

,productId,name,cleanName,imageUrl,categoryId,groupId,url,modifiedOn,imageCount,isPresale,...,Description,Level,Cost,CardType,Color,Trait,Link Condition,Attack Points,Hit Points,Date
0,662113,Steel Requiem Booster Pack,Steel Requiem Booster Pack,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662113/gunda...,2026-01-28T15:30:43.96,1,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
1,662114,Steel Requiem Booster Box,Steel Requiem Booster Box,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662114/gunda...,2026-01-28T15:30:16.87,1,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
2,662115,Steel Requiem Booster Box Case,Steel Requiem Booster Box Case,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662115/gunda...,2026-01-28T15:30:28.78,1,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
3,662117,Steel Requiem Sleeved Booster Pack,Steel Requiem Sleeved Booster Pack,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662117/gunda...,2025-11-04T17:08:09.637,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
4,670488,Gundam NT-1,Gundam NT 1,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670488/gunda...,2026-01-22T13:55:33.877,1,True,...,"[Repair 2] (At the end of your turn, this Unit...",5,4,Unit,Blue,(Earth Federation),[Christina Mackenzie] / [Amuro Ray],4,4,2026-01-29


In [6]:
price_df.head()

,productId,lowPrice,midPrice,highPrice,marketPrice,directLowPrice,subTypeName,Date
0,662114,138.47,140.90,299.00,135.56,None,Normal,2026-01-29
1,662115,1593.81,1634.35,2099.64,1704.16,None,Normal,2026-01-29
2,670488,18.65,20.00,30.00,18.12,None,Holofoil,2026-01-29
3,670489,20.00,25.00,90.99,24.00,None,Holofoil,2026-01-29
4,670490,0.32,5.50,50.32,0.32,None,Holofoil,2026-01-29


## Data Validation

In [7]:
price_df.isna().any()

productId         False
lowPrice          False
midPrice          False
highPrice         False
marketPrice        True
directLowPrice     True
subTypeName       False
Date              False
dtype: bool

In [8]:
price_df[price_df['marketPrice'].isna()]

,productId,lowPrice,midPrice,highPrice,marketPrice,directLowPrice,subTypeName,Date
7,670493,2.32,3.00,20.32,NaN,None,Holofoil,2026-01-29
8,670494,0.25,0.40,1.00,NaN,None,Normal,2026-01-29
9,670495,0.25,0.40,1.00,NaN,None,Normal,2026-01-29
17,670503,0.25,0.40,1.00,NaN,None,Normal,2026-01-29
23,670509,0.50,1.50,2.00,NaN,None,Normal,2026-01-29
24,670510,1.50,3.25,5.00,NaN,None,Normal,2026-01-29
25,670511,0.25,0.40,1.00,NaN,None,Normal,2026-01-29
26,670512,0.25,0.40,1.00,NaN,None,Normal,2026-01-29
32,670518,0.25,0.33,0.40,NaN,None,Normal,2026-01-29
36,670522,2.32,7.51,20.32,NaN,None,Holofoil,2026-01-29


In [9]:
#Get the ProductIDs of those missing a Market Price
missing_mp_list = []
for item in price_df[price_df['marketPrice'].isna()]['productId']:
    missing_mp_list.append(item)

In [10]:
product_df[product_df['productId'].isin(missing_mp_list) == True]

,productId,name,cleanName,imageUrl,categoryId,groupId,url,modifiedOn,imageCount,isPresale,...,Description,Level,Cost,CardType,Color,Trait,Link Condition,Attack Points,Hit Points,Date
9,670493,Penelope (Middle Form),Penelope Middle Form,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670493/gunda...,2025-12-24T20:07:36.657,1,True,...,[Deploy] Choose 1 to 2 enemy Units with 3 or l...,6,5,Unit,Blue,(Earth Federation),[Lane Aim],4,4,2026-01-29
10,670494,Hizack Custom,Hizack Custom,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670494/gunda...,2026-01-22T13:58:42.09,1,True,...,While you have 2 or more (Titans) Units in pla...,3,2,Unit,Blue,(Titans),NaN,3,2,2026-01-29
11,670495,Baund Doc,Baund Doc,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670495/gunda...,2025-12-24T20:07:48.96,1,True,...,[Activate: Main] [Once per Turn] Exile 3 (Tita...,6,5,Unit,Blue,(Titans),[Jerid Messa],4,5,2026-01-29
19,670503,Auda's Maganac,Audas Maganac,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670503/gunda...,2025-12-24T20:09:57.883,1,True,...,"[Attack] If you are attacking an enemy Unit, t...",3,2,Unit,Green,(Maganac Corps),(Maganac Corps) Trait,2,3,2026-01-29
25,670509,Patulia,Patulia,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670509/gunda...,2025-12-24T20:10:38.167,1,True,...,[Deploy] Deal 3 damage to all Bases.,7,6,Unit,Red,(SRA),[Carris Nautilus],5,6,2026-01-29
26,670510,Duel Gundam (Assault Shroud) (GD03-042),Duel Gundam Assault Shroud GD03 042,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670510/gunda...,2025-12-24T20:10:43.59,1,True,...,"While this Unit has 5 or more AP, it may choos...",5,3,Unit,Red,(ZAFT),[Yzak Joule],3,4,2026-01-29
27,670511,Daughtress Flyer,Daughtress Flyer,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670511/gunda...,2025-12-24T20:10:49.213,1,True,...,[Deploy] Deploy 1 rested [Daughtress]((New UNE...,3,2,Unit,Red,(New UNE),NaN,2,3,2026-01-29
28,670512,Balient,Balient,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670512/gunda...,2025-12-24T20:10:57.177,1,True,...,"While you have a Unit token in play, this Unit...",3,2,Unit,Red,(New UNE),NaN,2,4,2026-01-29
34,670518,Defurse,Defurse,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670518/gunda...,2025-12-24T20:11:44.183,1,True,...,[Deploy] You may 1 (X-Rounder) card from your ...,5,4,Unit,Purple,(UE) (Vagan),NaN,2,5,2026-01-29
38,670522,Aile Strike Gundam,Aile Strike Gundam,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/670522/gunda...,2025-12-24T20:12:11.457,1,True,...,[Blocker] (Rest this Unit to change the attack...,4,3,Unit,White,(Triple Ship Alliance),[Mu La Flaga],3,4,2026-01-29


<div class="alert alert-block alert-info">
    The products currently listed at the time of writing <b>(01/27)</b> are negligible. Some examples of the products listed are "Bonus Packs", which is a single holofoil card that comes with the purchase of each structure deck. The pack themselves are typically not sold individually, which explains why there is no set market price for them. The other examples include the "Premium Card Collection", which are exclusive to the Bandai Regional Card Fest event. Given the exclusivity of these cards, on top of needing to be selected via a lottery to purchase the collection, there is no set market for this card quite yet.
 
</div>

In [11]:
price_df['directLowPrice'].value_counts()

Series([], Name: directLowPrice, dtype: int64)

<div class="alert alert-block alert-info">
    Nulls and missing values are true in the <b>"marketPrice" and "directLowPrice"</b> column.
<br> 
    
This is to be expected given that TCGPlayer differentiates their own pricing:
* {directLowPrice} is the lowest price for a card in the TCGPlayer warehouse.
* {lowPrice} is the lowest price from any independent seller in the marketplace.

For Gundam TCG specifically, TCGPlayer **does not** directly sell any Gundam cards, leading to the null values that we have now.

In [12]:
product_df.isnull().any()

productId         False
name              False
cleanName         False
imageUrl          False
categoryId        False
groupId           False
url               False
modifiedOn        False
imageCount        False
isPresale         False
releasedOn         True
Rarity             True
Number             True
Description        True
Level              True
Cost               True
CardType           True
Color              True
Trait              True
Link Condition     True
Attack Points      True
Hit Points         True
Date              False
dtype: bool

In [13]:
product_df[product_df['Rarity'].isnull() == True]

,productId,name,cleanName,imageUrl,categoryId,groupId,url,modifiedOn,imageCount,isPresale,...,Description,Level,Cost,CardType,Color,Trait,Link Condition,Attack Points,Hit Points,Date
0,662113,Steel Requiem Booster Pack,Steel Requiem Booster Pack,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662113/gunda...,2026-01-28T15:30:43.96,1,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
1,662114,Steel Requiem Booster Box,Steel Requiem Booster Box,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662114/gunda...,2026-01-28T15:30:16.87,1,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
2,662115,Steel Requiem Booster Box Case,Steel Requiem Booster Box Case,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662115/gunda...,2026-01-28T15:30:28.78,1,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
3,662117,Steel Requiem Sleeved Booster Pack,Steel Requiem Sleeved Booster Pack,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662117/gunda...,2025-11-04T17:08:09.637,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
201,675730,Battle of Aces (SP) (R+),Battle of Aces SP Rplus,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/675730/gunda...,2026-01-29T19:50:01.367,0,True,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
988,671153,Store Tournament Winner Pack 02,Store Tournament Winner Pack 02,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24340,https://www.tcgplayer.com/product/671153/gunda...,2026-01-06T17:59:32.34,0,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
989,671155,Store Tournament Participation Pack 03,Store Tournament Participation Pack 03,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24340,https://www.tcgplayer.com/product/671155/gunda...,2026-01-06T18:00:15.723,1,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
990,671158,Store Tournament Winner Pack 03,Store Tournament Winner Pack 03,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24340,https://www.tcgplayer.com/product/671158/gunda...,2026-01-06T18:01:03.167,1,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29
1114,617436,Gundam Card Game Edition Beta Box,Gundam Card Game Edition Beta Box,https://tcgplayer-cdn.tcgplayer.com/product/61...,86,24193,https://www.tcgplayer.com/product/617436/gunda...,2025-02-07T19:03:43.763,1,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-01-29


## Data Transformation

In [14]:
price_df.shape

(1045, 8)

In [15]:
product_df.shape

(1116, 23)

In [16]:
#Join the data for easier viewing
df = pd.merge(product_df,price_df,how='left',left_on='productId', right_on='productId')

<div class="alert alert-block alert-info">
    <b> As of January 27th, </b> the Product_df has more data rows as a result of having active "Product Display Pages" (PDP) for the GD03 Booster Set. As this set is due to be released on Friday, January 30th, there is no active pricing data to join to the product information, leading to the null values seen in the columns joined from the pricing_df. 
    
> __For future proofing of this project, a left join will be conducted rather than an inner to enable us to still actively view card details.__
 
</div>

In [17]:
df.shape

(1117, 30)

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1117 entries, 0 to 1116
Data columns (total 30 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   productId       1117 non-null   int64  
 1   name            1117 non-null   object 
 2   cleanName       1117 non-null   object 
 3   imageUrl        1117 non-null   object 
 4   categoryId      1117 non-null   int64  
 5   groupId         1117 non-null   int64  
 6   url             1117 non-null   object 
 7   modifiedOn      1117 non-null   object 
 8   imageCount      1117 non-null   int64  
 9   isPresale       1117 non-null   bool   
 10  releasedOn      724 non-null    object 
 11  Rarity          1046 non-null   object 
 12  Number          1046 non-null   object 
 13  Description     902 non-null    object 
 14  Level           918 non-null    object 
 15  Cost            918 non-null    object 
 16  CardType        1043 non-null   object 
 17  Color           918 non-null    o

In [19]:
df.isnull().any()

productId         False
name              False
cleanName         False
imageUrl          False
categoryId        False
groupId           False
url               False
modifiedOn        False
imageCount        False
isPresale         False
releasedOn         True
Rarity             True
Number             True
Description        True
Level              True
Cost               True
CardType           True
Color              True
Trait              True
Link Condition     True
Attack Points      True
Hit Points         True
Date_x            False
lowPrice           True
midPrice           True
highPrice          True
marketPrice        True
directLowPrice     True
subTypeName        True
Date_y             True
dtype: bool

<alert alert-block alert-info>

In [20]:
df[df['lowPrice'].isnull() == True]

,productId,name,cleanName,imageUrl,categoryId,groupId,url,modifiedOn,imageCount,isPresale,...,Attack Points,Hit Points,Date_x,lowPrice,midPrice,highPrice,marketPrice,directLowPrice,subTypeName,Date_y
0,662113,Steel Requiem Booster Pack,Steel Requiem Booster Pack,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662113/gunda...,2026-01-28T15:30:43.96,1,True,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,662117,Steel Requiem Sleeved Booster Pack,Steel Requiem Sleeved Booster Pack,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,24522,https://www.tcgplayer.com/product/662117/gunda...,2025-11-04T17:08:09.637,0,True,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
154,675678,A Healthy Curiosity (R+),A Healthy Curiosity Rplus,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/675678/gunda...,2026-01-29T18:43:26.853,0,True,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
155,675679,A Show of Resolve (SP) (U+),A Show of Resolve SP Uplus,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/675679/gunda...,2026-01-29T18:43:26.9,0,True,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
156,675680,Altron Gundam (LR+),Altron Gundam LRplus,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24522,https://www.tcgplayer.com/product/675680/gunda...,2026-01-29T18:43:26.95,0,True,...,5,6,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
984,670595,Gundam Lfrith (Premium Card Collection),Gundam Lfrith Premium Card Collection,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24340,https://www.tcgplayer.com/product/670595/gunda...,2026-01-05T20:09:50.227,1,False,...,2,4,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
985,670968,Premium Card Collection 01 [EVX05],Premium Card Collection 01 EVX05,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24340,https://www.tcgplayer.com/product/670968/gunda...,2026-01-05T20:18:03.993,1,False,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
995,675653,Zaku II FZ (GD03 Release Event),Zaku II FZ GD03 Release Event,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24340,https://www.tcgplayer.com/product/675653/gunda...,2026-01-29T16:17:50.93,1,False,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
996,675654,Bernard Wiseman (GD03 Release Event),Bernard Wiseman GD03 Release Event,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,24340,https://www.tcgplayer.com/product/675654/gunda...,2026-01-29T16:18:56.987,1,False,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
#Clean up HTML formatting from text
remove_strings = ['<br>','\r','\n']
for i in df['Description']:
    for string in remove_strings:
        df['Description'] = df['Description'].str.replace(string,' ')

In [22]:
df[['name','Description']]

,name,Description
0,Steel Requiem Booster Pack,NaN
1,Steel Requiem Booster Box,NaN
2,Steel Requiem Booster Box Case,NaN
3,Steel Requiem Sleeved Booster Pack,NaN
4,Gundam NT-1,"[Repair 2] (At the end of your turn, this Unit..."
...,...,...
1112,Resource (C+),(Rest a Resource when paying a cost.)
1113,EX Base,"(At the start of the game, place 1 active EX B..."
1114,EX Resource,"(At the start of the game, the second-turn pla..."
1115,Gundam Card Game Edition Beta Box,NaN


In [23]:
#Rename columns for easier readability
df.rename(columns = {
    'groupId':'Set',
    'subTypeName':'Holofoil',
    'Date_x':'Date'
},inplace=True)

In [24]:
set_dict = {
    24193:'Edition Beta',
    24221: 'Newtype Rising (GD01)',
    24222: 'Starter Deck 01: Heroic Beginnings (ST01)',
    24223: 'Starter Deck 02: Wings of Advance (ST02)',
    24224: 'Starter Deck 03: Zeon\'s Rush (ST03)',
    24225: 'Starter Deck 04: SEED Strike (ST04)',
    24340: 'Gundam Promotional Cards',
    24372: 'Promotional Resource Tokens',
    24373: 'Promotional EX Resource Tokens',
    24374: 'Promotional EX Base Tokens',
    24407: 'Starter Deck 05: Iron Bloom (ST05)',
    24408: 'Dual Impact (GD02)',
    24409: 'Starter Deck 06: Clan Unity (ST06)',
    24410: 'Starter Deck 07: Celestial Drive (ST07)',
    24411: 'Starter Deck 08: Flash of Radiance (ST08)',
    24522: 'Steel Requiem (GD03)',
}

#alternative
#df['Set'] = df['Set'].replace(set_dict) 

for x,y in set_dict.items():
    df.loc[df['Set'] == x,'Set'] = y
    
df['Set'].value_counts()

Steel Requiem (GD03)                         204
Newtype Rising (GD01)                        197
Dual Impact (GD02)                           191
Gundam Promotional Cards                     105
Edition Beta                                  85
Starter Deck 01: Heroic Beginnings (ST01)     39
Starter Deck 02: Wings of Advance (ST02)      39
Starter Deck 03: Zeon's Rush (ST03)           39
Starter Deck 04: SEED Strike (ST04)           39
Starter Deck 07: Celestial Drive (ST07)       36
Starter Deck 08: Flash of Radiance (ST08)     36
Starter Deck 06: Clan Unity (ST06)            36
Starter Deck 05: Iron Bloom (ST05)            36
Promotional Resource Tokens                   21
Promotional EX Base Tokens                    11
Promotional EX Resource Tokens                 3
Name: Set, dtype: int64

In [25]:
#Export today's data to a CSV and proceed to exploration!
df.to_csv(f'Gundam TCG Pricing Raw Data/gundam_pricing_{str(date)}.csv',index=False)

In [26]:
#Add data to the master csv for storage
import glob
import os
all_files = glob.glob(os.path.join('Gundam TCG Pricing Raw Data', "gundam_pricing*.csv"))
csv_list = []
for csv in all_files:
    read = pd.read_csv(csv)
    csv_list.append(read)

combined_csvs = pd.concat(csv_list,ignore_index=True)
combined_csvs.to_csv('Gundam TCG Pricing Raw Data/master_pricing.csv',index=False)

In [27]:
check = pd.read_csv('Gundam TCG Pricing Raw Data/master_pricing.csv')
check.isnull().any()

productId         False
name              False
cleanName         False
imageUrl          False
categoryId        False
Set               False
url               False
modifiedOn        False
imageCount        False
isPresale         False
releasedOn         True
Rarity             True
Number             True
Description        True
Level              True
Cost               True
CardType           True
Color              True
Trait              True
Link Condition     True
Attack Points      True
Hit Points         True
Date              False
lowPrice           True
midPrice           True
highPrice          True
marketPrice        True
directLowPrice     True
Holofoil           True
Date_y             True
dtype: bool

# Data Exploration

In [28]:
#Expand Rows for viewing
pd.set_option('display.max.rows', None)

In [29]:
df[df['Set'].str.contains('Steel Requiem')]

,productId,name,cleanName,imageUrl,categoryId,Set,url,modifiedOn,imageCount,isPresale,...,Attack Points,Hit Points,Date,lowPrice,midPrice,highPrice,marketPrice,directLowPrice,Holofoil,Date_y
0,662113,Steel Requiem Booster Pack,Steel Requiem Booster Pack,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/662113/gunda...,2026-01-28T15:30:43.96,1,True,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,662114,Steel Requiem Booster Box,Steel Requiem Booster Box,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/662114/gunda...,2026-01-28T15:30:16.87,1,True,...,NaN,NaN,2026-01-29,138.47,140.90,299.00,135.56,None,Normal,2026-01-29
2,662115,Steel Requiem Booster Box Case,Steel Requiem Booster Box Case,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/662115/gunda...,2026-01-28T15:30:28.78,1,True,...,NaN,NaN,2026-01-29,1593.81,1634.35,2099.64,1704.16,None,Normal,2026-01-29
3,662117,Steel Requiem Sleeved Booster Pack,Steel Requiem Sleeved Booster Pack,https://tcgplayer-cdn.tcgplayer.com/product/66...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/662117/gunda...,2025-11-04T17:08:09.637,0,True,...,NaN,NaN,2026-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,670488,Gundam NT-1,Gundam NT 1,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/670488/gunda...,2026-01-22T13:55:33.877,1,True,...,4,4,2026-01-29,18.65,20.00,30.00,18.12,None,Holofoil,2026-01-29
5,670489,The-O,The O,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/670489/gunda...,2025-12-24T20:07:10.957,1,True,...,5,5,2026-01-29,20.00,25.00,90.99,24.00,None,Holofoil,2026-01-29
6,670490,Messala,Messala,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/670490/gunda...,2026-01-22T14:03:01.83,1,True,...,5,4,2026-01-29,0.32,5.50,50.32,0.32,None,Holofoil,2026-01-29
7,670491,Hambrabi,Hambrabi,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/670491/gunda...,2026-01-22T13:58:15.99,1,True,...,5,4,2026-01-29,1.32,3.50,8.00,1.32,None,Holofoil,2026-01-29
8,670492,Kshatriya Besserung,Kshatriya Besserung,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/670492/gunda...,2026-01-22T14:02:46.58,1,True,...,4,4,2026-01-29,1.00,5.50,20.32,1.00,None,Holofoil,2026-01-29
9,670493,Penelope (Middle Form),Penelope Middle Form,https://tcgplayer-cdn.tcgplayer.com/product/67...,86,Steel Requiem (GD03),https://www.tcgplayer.com/product/670493/gunda...,2025-12-24T20:07:36.657,1,True,...,4,4,2026-01-29,2.32,3.00,20.32,NaN,None,Holofoil,2026-01-29


In [30]:
#Drop unncessary columns for exploration
df_copy = df.copy() #back-up
df.drop(columns=['productId',
                 'cleanName',
                 'categoryId',
                 'modifiedOn',
                 'isPresale',
                 'releasedOn',
                 'directLowPrice',
                 'imageCount',
                 'Date_y',
                 'Number',
                 'Description',
                 'Level',
                 'Cost',
                 'Trait',
                 'Link Condition',
                 'Attack Points',
                 'Hit Points'
                ]
        ,inplace=True)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1117 entries, 0 to 1116
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   name         1117 non-null   object 
 1   imageUrl     1117 non-null   object 
 2   Set          1117 non-null   object 
 3   url          1117 non-null   object 
 4   Rarity       1046 non-null   object 
 5   CardType     1043 non-null   object 
 6   Color        918 non-null    object 
 7   Date         1117 non-null   object 
 8   lowPrice     1045 non-null   float64
 9   midPrice     1045 non-null   float64
 10  highPrice    1045 non-null   float64
 11  marketPrice  1002 non-null   float64
 12  Holofoil     1045 non-null   object 
dtypes: float64(4), object(9)
memory usage: 122.2+ KB


In [32]:
#Keep only card information
case_mask = ~df['name'].str.contains('Starter Deck|Display Case|Booster Box Case|Booster Pack|Booster Box')

In [33]:
df = df[case_mask]

In [34]:
#What is the most expensive card(s) currently listed at?
df[df['marketPrice'] == df['marketPrice'].max()]

,name,imageUrl,Set,url,Rarity,CardType,Color,Date,lowPrice,midPrice,highPrice,marketPrice,Holofoil
1046,Wing Gundam (LR+),https://tcgplayer-cdn.tcgplayer.com/product/61...,Edition Beta,https://www.tcgplayer.com/product/616611/gunda...,LR+,Unit,Green,2026-01-29,5000.0,5000.0,5000.0,3999.99,Holofoil


In [35]:
#What is the current average market price of cards per set?
df.groupby('Set')['marketPrice'].mean().round(2).sort_values(ascending=False)

Set
Edition Beta                                 161.16
Promotional EX Resource Tokens               105.26
Gundam Promotional Cards                      45.85
Promotional EX Base Tokens                    29.36
Promotional Resource Tokens                   25.07
Starter Deck 07: Celestial Drive (ST07)       12.33
Newtype Rising (GD01)                         10.25
Starter Deck 04: SEED Strike (ST04)           10.02
Starter Deck 08: Flash of Radiance (ST08)      9.62
Starter Deck 01: Heroic Beginnings (ST01)      8.82
Starter Deck 05: Iron Bloom (ST05)             8.61
Dual Impact (GD02)                             8.24
Starter Deck 03: Zeon's Rush (ST03)            8.21
Starter Deck 06: Clan Unity (ST06)             7.79
Starter Deck 02: Wings of Advance (ST02)       7.46
Steel Requiem (GD03)                           3.09
Name: marketPrice, dtype: float64

In [36]:
#What is the price gap between rarities?
df.groupby('Rarity')['marketPrice'].mean().round(2).sort_values(ascending=False)

Rarity
LR++           393.76
U+             349.52
LR+            176.89
R+              39.01
Promo           33.29
C++             29.47
Legend Rare     25.49
C+              20.49
Common           5.82
Rare             5.04
Uncommon         1.03
Name: marketPrice, dtype: float64

In [37]:
df[df['Rarity'] == 'LR++'].sort_values(by='marketPrice',ascending=False)

,name,imageUrl,Set,url,Rarity,CardType,Color,Date,lowPrice,midPrice,highPrice,marketPrice,Holofoil
727,Gundam (GD01-001) (LR++),https://tcgplayer-cdn.tcgplayer.com/product/64...,Newtype Rising (GD01),https://www.tcgplayer.com/product/645375/gunda...,LR++,Unit,Blue,2026-01-29,699.99,710.0,805.00,471.05,Holofoil
728,Gundam Aerial Rebuild (LR++),https://tcgplayer-cdn.tcgplayer.com/product/64...,Newtype Rising (GD01),https://www.tcgplayer.com/product/645376/gunda...,LR++,Unit,White,2026-01-29,428.99,504.0,999.00,427.73,Holofoil
372,Gundam Barbatos 1st Form (LR++),https://tcgplayer-cdn.tcgplayer.com/product/65...,Dual Impact (GD02),https://www.tcgplayer.com/product/659292/gunda...,LR++,Unit,Purple,2026-01-29,380.00,450.0,460.81,390.38,Holofoil
359,GQuuuuuuX (Omega Psycommu) (LR++),https://tcgplayer-cdn.tcgplayer.com/product/65...,Dual Impact (GD02),https://www.tcgplayer.com/product/659279/gunda...,LR++,Unit,Red,2026-01-29,269.90,310.0,499.99,285.89,Holofoil


In [38]:
#Price Gap between the LR++ (highest rarity) cards?
max_rarity_max_price = df[df['Rarity'] == 'LR++']['marketPrice'].max()
max_rarity_low_price = df[df['Rarity'] == 'LR++']['marketPrice'].min()
max_rarity_max_price - max_rarity_low_price

185.16000000000003

In [39]:
#What is the most expensive card by set?
set_max_card = df.groupby('Set')['marketPrice'].transform(max)
max_per_set = df.loc[df['marketPrice'] == set_max_card,['name','Set','Rarity','marketPrice','Holofoil']].sort_values(by='marketPrice',ascending=False)

In [40]:
max_per_set

,name,Set,Rarity,marketPrice,Holofoil
1046,Wing Gundam (LR+),Edition Beta,LR+,3999.99,Holofoil
919,Heero Yuy (Championship Finalist Card 01),Gundam Promotional Cards,Common,1600.00,Holofoil
727,Gundam (GD01-001) (LR++),Newtype Rising (GD01),LR++,471.05,Holofoil
372,Gundam Barbatos 1st Form (LR++),Dual Impact (GD02),LR++,390.38,Holofoil
1010,EX Resource (EXRP-003) (Mobile Suit Gundam the...,Promotional EX Resource Tokens,Promo,259.09,Holofoil
997,EX Base (EXBP-001) (Mobile Suit Gundam),Promotional EX Base Tokens,Promo,175.43,Holofoil
875,Aile Strike Gundam (LR+),Starter Deck 04: SEED Strike (ST04),LR+,99.32,Holofoil
234,Gundam Dynames (LR+),Starter Deck 07: Celestial Drive (ST07),LR+,80.34,Holofoil
758,Gundam (LR+),Starter Deck 01: Heroic Beginnings (ST01),LR+,79.52,Holofoil
836,Sinanju (LR+),Starter Deck 03: Zeon's Rush (ST03),LR+,75.36,Holofoil


In [41]:
#What is the most expensive 'color' card by set?
green_df = df[df['Color'] == 'Green'][['name','Set','CardType','Color','marketPrice','Holofoil']]
white_df = df[df['Color'] == 'White'][['name','Set','CardType','Color','marketPrice','Holofoil']]
red_df = df[df['Color'] == 'Red'][['name','Set','CardType','Color','marketPrice','Holofoil']]
blue_df = df[df['Color'] == 'Blue'][['name','Set','CardType','Color','marketPrice','Holofoil']]
purple_df = df[df['Color'] == 'Purple'][['name','Set','CardType','Color','marketPrice','Holofoil']]

In [42]:
#Green
green_set_max = green_df.groupby('Set')['marketPrice'].transform(max)
green_df[green_df['marketPrice'] == green_set_max].sort_values(by='marketPrice',ascending=False)

,name,Set,CardType,Color,marketPrice,Holofoil
1046,Wing Gundam (LR+),Edition Beta,Unit,Green,3999.99,Holofoil
919,Heero Yuy (Championship Finalist Card 01),Gundam Promotional Cards,Pilot,Green,1600.00,Holofoil
234,Gundam Dynames (LR+),Starter Deck 07: Celestial Drive (ST07),Unit,Green,80.34,Holofoil
696,Wing Gundam Zero (LR+),Newtype Rising (GD01),Unit,Green,76.84,Holofoil
797,Wing Gundam (LR+),Starter Deck 02: Wings of Advance (ST02),Unit,Green,69.95,Holofoil
841,Char's Zaku II (LR+),Starter Deck 03: Zeon's Rush (ST03),Unit,Green,44.04,Holofoil
480,Red Gundam (LR+),Starter Deck 06: Clan Unity (ST06),Unit,Green,41.06,Holofoil
13,Altron Gundam,Steel Requiem (GD03),Unit,Green,27.56,Holofoil
405,Flit Asuno (R+),Dual Impact (GD02),Pilot,Green,25.21,Holofoil


In [43]:
green_df.sort_values(by='marketPrice',ascending=False)

,name,Set,CardType,Color,marketPrice,Holofoil
1046,Wing Gundam (LR+),Edition Beta,Unit,Green,3999.99,Holofoil
919,Heero Yuy (Championship Finalist Card 01),Gundam Promotional Cards,Pilot,Green,1600.00,Holofoil
1056,Char Aznable (C+),Edition Beta,Pilot,Green,595.11,Holofoil
1076,Char's Zaku II (R+),Edition Beta,Unit,Green,552.14,Holofoil
1101,First Contact (U+),Edition Beta,Command,Green,452.75,Normal
234,Gundam Dynames (LR+),Starter Deck 07: Celestial Drive (ST07),Unit,Green,80.34,Holofoil
696,Wing Gundam Zero (LR+),Newtype Rising (GD01),Unit,Green,76.84,Holofoil
797,Wing Gundam (LR+),Starter Deck 02: Wings of Advance (ST02),Unit,Green,69.95,Holofoil
930,Shenlong Gundam (GD01-029) (Judge Pack 01),Gundam Promotional Cards,Unit,Green,56.75,Holofoil
969,Gundam Deathscythe Hell (Store Tournament Winn...,Gundam Promotional Cards,Unit,Green,50.84,Holofoil


In [44]:
#White
white_set_max = white_df.groupby('Set')['marketPrice'].transform(max)
white_df[white_df['marketPrice'] == white_set_max].sort_values(by='marketPrice',ascending=False)

,name,Set,CardType,Color,marketPrice,Holofoil
1104,Overflowing Affection (U+),Edition Beta,Command,White,2385.97,Holofoil
728,Gundam Aerial Rebuild (LR++),Newtype Rising (GD01),Unit,White,427.73,Holofoil
910,Gundam Aerial Rebuild (Newtype Challenge 2025 ...,Gundam Promotional Cards,Unit,White,328.16,Holofoil
875,Aile Strike Gundam (LR+),Starter Deck 04: SEED Strike (ST04),Unit,White,99.32,Holofoil
766,Zowort (C+),Starter Deck 01: Heroic Beginnings (ST01),Unit,White,28.95,Holofoil
36,Graham's Union Flag Custom,Steel Requiem (GD03),Unit,White,26.84,Holofoil
387,Zeta Gundam (LR+),Dual Impact (GD02),Unit,White,22.50,Holofoil
529,McGillis' Schwalbe Graze (LR+),Starter Deck 05: Iron Bloom (ST05),Unit,White,16.14,Holofoil


In [45]:
#Red
red_set_max = red_df.groupby('Set')['marketPrice'].transform(max)
red_df[red_df['marketPrice'] == red_set_max].sort_values(by='marketPrice',ascending=False)

,name,Set,CardType,Color,marketPrice,Holofoil
959,Qubeley (Newtype Challenge 2025 Mission 3),Gundam Promotional Cards,Unit,Red,360.64,Holofoil
359,GQuuuuuuX (Omega Psycommu) (LR++),Dual Impact (GD02),Unit,Red,285.89,Holofoil
836,Sinanju (LR+),Starter Deck 03: Zeon's Rush (ST03),Unit,Red,75.36,Holofoil
715,Marida Cruz (R+),Newtype Rising (GD01),Pilot,Red,67.28,Holofoil
259,Xi Gundam (ST08-001) (LR+),Starter Deck 08: Flash of Radiance (ST08),Unit,Red,54.68,Holofoil
475,GQuuuuuuX (Omega Psycommu) (LR+),Starter Deck 06: Clan Unity (ST06),Unit,Red,44.65,Holofoil
880,Aegis Gundam (LR+),Starter Deck 04: SEED Strike (ST04),Unit,Red,36.76,Holofoil
81,Providence Gundam,Steel Requiem (GD03),Unit,Red,24.01,Holofoil


In [46]:
#Blue
blue_set_max = blue_df.groupby('Set')['marketPrice'].transform(max)
blue_df[blue_df['marketPrice'] == blue_set_max].sort_values(by='marketPrice',ascending=False)

,name,Set,CardType,Color,marketPrice,Holofoil
1033,Gundam (LR+),Edition Beta,Unit,Blue,1090.26,Holofoil
727,Gundam (GD01-001) (LR++),Newtype Rising (GD01),Unit,Blue,471.05,Holofoil
758,Gundam (LR+),Starter Deck 01: Heroic Beginnings (ST01),Unit,Blue,79.52,Holofoil
328,Gundam Epyon (LR+),Dual Impact (GD02),Unit,Blue,72.87,Holofoil
271,Penelope (LR+),Starter Deck 08: Flash of Radiance (ST08),Unit,Blue,60.39,Holofoil
924,Guntank (Championship Participation Pack 01),Gundam Promotional Cards,Unit,Blue,53.84,Holofoil
802,Tallgeese (LR+),Starter Deck 02: Wings of Advance (ST02),Unit,Blue,34.29,Holofoil
5,The-O,Steel Requiem (GD03),Unit,Blue,24.00,Holofoil


In [47]:
#Purple
purple_set_max = purple_df.groupby('Set')['marketPrice'].transform(max)
purple_df[purple_df['marketPrice'] == purple_set_max].sort_values(by='marketPrice',ascending=False)

,name,Set,CardType,Color,marketPrice,Holofoil
372,Gundam Barbatos 1st Form (LR++),Dual Impact (GD02),Unit,Purple,390.38,Holofoil
224,Gundam Exia (ST07-001) (LR+),Starter Deck 07: Celestial Drive (ST07),Unit,Purple,80.20,Holofoil
523,Gundam Barbatos 4th Form (LR+),Starter Deck 05: Iron Bloom (ST05),Unit,Purple,67.74,Holofoil
981,Gundam Barbatos 2nd Form (Premium Card Collect...,Gundam Promotional Cards,Unit,Purple,45.00,Holofoil
29,Gundam Exia (Trans-Am),Steel Requiem (GD03),Unit,Purple,27.45,Holofoil


In [50]:
df[df['Set'].str.contains('Steel Requiem')].sort_values(by='marketPrice',ascending=False)

,name,imageUrl,Set,url,Rarity,CardType,Color,Date,lowPrice,midPrice,highPrice,marketPrice,Holofoil
13,Altron Gundam,https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/670497/gunda...,Legend Rare,Unit,Green,2026-01-29,23.68,33.00,35.00,27.56,Holofoil
29,Gundam Exia (Trans-Am),https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/670513/gunda...,Legend Rare,Unit,Purple,2026-01-29,34.67,35.50,50.00,27.45,Holofoil
36,Graham's Union Flag Custom,https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/670520/gunda...,Legend Rare,Unit,White,2026-01-29,15.01,44.99,90.99,26.84,Holofoil
81,Providence Gundam,https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/673466/gunda...,Legend Rare,Unit,Red,2026-01-29,20.00,37.00,90.99,24.01,Holofoil
5,The-O,https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/670489/gunda...,Legend Rare,Unit,Blue,2026-01-29,20.00,25.00,90.99,24.00,Holofoil
37,Freedom Gundam (GD03-070),https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/670521/gunda...,Legend Rare,Unit,White,2026-01-29,23.68,39.99,50.00,23.94,Holofoil
75,Gundam AGE-2 Normal,https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/673460/gunda...,Legend Rare,Unit,Green,2026-01-29,20.00,25.00,90.99,23.01,Holofoil
30,Gundam Barbatos Lupus,https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/670514/gunda...,Legend Rare,Unit,Purple,2026-01-29,16.68,24.95,35.00,19.62,Holofoil
4,Gundam NT-1,https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/670488/gunda...,Legend Rare,Unit,Blue,2026-01-29,18.65,20.00,30.00,18.12,Holofoil
82,GQuuuuuuX (Omega Psycommu),https://tcgplayer-cdn.tcgplayer.com/product/67...,Steel Requiem (GD03),https://www.tcgplayer.com/product/673467/gunda...,Legend Rare,Unit,Red,2026-01-29,13.68,39.47,65.00,14.47,Holofoil
